In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder

In [7]:
data = pd.read_csv('data/taxi_trip_pricing_sub.csv')
data = data.dropna(ignore_index=True)

categorical_columns = ['Time_of_Day', 'Day_of_Week', 'Traffic_Conditions', 'Weather']
numerical_columns = ['Trip_Distance_km', 'Passenger_Count', 'Base_Fare','Trip_Duration_Minutes']
target_col = 'Trip_Price'

target = data[target_col]

encoder = OneHotEncoder(sparse_output=False)
encoder.fit(data[categorical_columns])
encoded_array = encoder.transform(data[categorical_columns])

encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(categorical_columns), index=data.index)
encoded_df

features = data.drop(columns=categorical_columns + [target_col]).join(encoded_df)

In [ ]:
# Train the regression decision tree on the full dataset
reg_tree = DecisionTreeRegressor(max_depth=1, random_state=42)
reg_tree.fit(features, target)

# Plot the regression tree
plt.figure(figsize=(12, 8))
plot_tree(reg_tree, feature_names=features.columns, filled=True)

plt.title("Regression Decision Tree")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# --- Generate synthetic data ---
np.random.seed(42)
X = np.linspace(0, 10, 100)[:, np.newaxis]
y = np.sin(X).ravel() + np.random.normal(scale=0.1, size=100)

# --- Fit Gradient Boosting Regressor ---
n_estimators = 10
gbr = GradientBoostingRegressor(n_estimators=n_estimators,
                                learning_rate=0.2,
                                max_depth=3,
                                random_state=42)
gbr.fit(X, y)

# --- Initial cumulative prediction from the baseline estimator ---
cum_pred = gbr.init_.predict(X)

# --- Determine a common y-scale for all plots ---
# We set the limits based on the range of y and the initial prediction, with a margin.
y_global_min = min(y.min(), cum_pred.min()) - 0.5
y_global_max = max(y.max(), cum_pred.max()) + 0.5

# --- Step-by-step visualization for each boosting stage ---
for i in range(n_estimators):
    # New learner's prediction (fitted on the residual)
    tree = gbr.estimators_[i, 0]
    learner_pred = gbr.learning_rate * tree.predict(X)
    learner_pred_raw = tree.predict(X)
    
    # Updated cumulative prediction after adding the new learner
    new_cum_pred = cum_pred + learner_pred
    
    # Residual (true value - new cumulative prediction)
    residual = y - new_cum_pred
    
    # Create a figure with 3 subplots (left: cumulative, middle: residual, right: new model)
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    
    # Left Plot: Cumulative Prediction
    axs[0].scatter(X, y, color='black', s=20, label='Data')
    axs[0].plot(X, new_cum_pred, color='green', label='Cumulative Prediction')
    axs[0].set_title(f"Stage {i+1}: Cumulative Prediction")
    axs[0].set_xlabel("Feature X")
    axs[0].set_ylabel("Target / Prediction")
    axs[0].legend()
    axs[0].grid(True)
    axs[0].set_ylim([y_global_min, y_global_max])
    
    # Middle Plot: Residuals
    axs[1].scatter(X, residual, color='blue', s=20, label='Residual (y - Prediction)')
    axs[1].hlines(0, X.min(), X.max(), colors='gray', linestyles='--')
    axs[1].set_title(f"Stage {i+1}: Residuals")
    axs[1].set_xlabel("Feature X")
    axs[1].set_ylabel("Residual")
    axs[1].legend()
    axs[1].grid(True)
    axs[1].set_ylim([y_global_min, y_global_max])
    
    # Right Plot: New Model (Learner's) Prediction on the Residual
    axs[2].plot(X, learner_pred_raw, color='red', linestyle='--', label="New Model's Prediction")
    axs[2].set_title(f"Stage {i+1}: New Model on Residual")
    axs[2].set_xlabel("Feature X")
    axs[2].set_ylabel("Learner Prediction")
    axs[2].legend()
    axs[2].grid(True)
    axs[2].set_ylim([y_global_min, y_global_max])
    
    plt.tight_layout()
    plt.show()
    plt.pause(2)  # Pause for 2 seconds to inspect the plots
    plt.close(fig)
    
    # Update cumulative prediction for the next stage
    cum_pred = new_cum_pred.copy()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# --- Generate synthetic data ---
np.random.seed(42)
X = np.linspace(0, 10, 100)[:, np.newaxis]
y = np.sin(X).ravel()*1.5 + np.random.normal(scale=0.1, size=100)

# --- Fit Gradient Boosting Regressor ---
n_estimators = 40
gbr = GradientBoostingRegressor(n_estimators=n_estimators,
                                learning_rate=0.2,
                                max_depth=2,
                                random_state=42)
gbr.fit(X, y)

# --- Initial cumulative prediction from the baseline estimator ---
# Note: This initial prediction is NOT multiplied by the learning rate.
cum_pred = gbr.init_.predict(X)
initial_mse = mean_squared_error(y, cum_pred)
print(f"Initial MSE: {initial_mse:.4f}")

# --- Determine a common y-scale for all plots ---
y_global_min = min(y.min(), cum_pred.min()) - 0.5
y_global_max = max(y.max(), cum_pred.max()) + 0.5

# --- Step-by-step visualization for each boosting stage ---
for i in range(n_estimators):
    # Residual that the new learner is trying to fit (before update)
    residual_to_fit = y - cum_pred
    
    # New learner's prediction (scaled by the learning rate)
    tree = gbr.estimators_[i, 0]
    # if i!=0:
    #     learner_pred = gbr.learning_rate * tree.predict(X)
    # else:
    #     learner_pred = tree.predict(X)
        
    learner_pred = gbr.learning_rate * tree.predict(X)
        
    
    # Updated cumulative prediction after adding the new learner
    new_cum_pred = cum_pred + learner_pred
    
    # Residual after update (true value minus new cumulative prediction)
    updated_residual = y - new_cum_pred
    
    resid_tree = gbr.estimators_[i+1, 0]
    resid_tree_pred = resid_tree.predict(X)
    
    
    # Create a figure with 3 subplots
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    
    # Left Plot: Cumulative Prediction after update
    axs[0].scatter(X, y, color='black', s=20, label='Data')
    axs[0].plot(X, cum_pred, color='green', label='Cumulative Prediction')
    axs[0].set_title(f"Stage {i+1}: Cumulative Prediction\nMSE: {mean_squared_error(y, new_cum_pred):.4f}")
    axs[0].set_xlabel("Feature X")
    axs[0].set_ylabel("Target / Prediction")
    axs[0].legend()
    axs[0].grid(True)
    axs[0].set_ylim([y_global_min, y_global_max])
    
    # Middle Plot: Residual after update
    axs[1].scatter(X, updated_residual, color='blue', s=20, label='Residual after update')
    axs[1].hlines(0, X.min(), X.max(), colors='gray', linestyles='--')
    axs[1].set_title(f"Stage {i+1}: Residual after Update")
    axs[1].set_xlabel("Feature X")
    axs[1].set_ylabel("Residual")
    axs[1].legend()
    axs[1].grid(True)
    axs[1].set_ylim([y_global_min, y_global_max])
    
    # Right Plot: New learner's fit on residual (scatter of residual & fitted line)
    axs[2].scatter(X, updated_residual, color='blue', s=20, label='Residual to Fit')
    axs[2].plot(X, resid_tree_pred, color='red', linestyle='--', label="Learner's Prediction")
    axs[2].set_title(f"Stage {i+1}: Learner's Fit on Residual")
    axs[2].set_xlabel("Feature X")
    axs[2].set_ylabel("Residual / Prediction")
    axs[2].legend()
    axs[2].grid(True)
    axs[2].set_ylim([y_global_min, y_global_max])
    
    plt.tight_layout()
    plt.show()
    plt.pause(2)  # Pause for 2 seconds to inspect the plots
    plt.close(fig)
    
    # Update cumulative prediction for the next stage
    cum_pred = new_cum_pred.copy()


In [2]:
import pandas as pd

In [4]:
df = pd.read_csv('/Users/dalibor/VsCodeProjects/imggi-course/notebooks/ml/data/UCI_Credit_Card.csv')

In [6]:
df2 = pd.read_csv('/Users/dalibor/VsCodeProjects/imggi-course/notebooks/ml/data/UCI_Credit_Card_600_samples.csv')

In [7]:
cls = df2.columns

In [8]:
df = df[cls]

In [9]:
df

,LIMIT_BAL,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,20000.0,24,2,2,-1,-1,-2,-2,3913.0,3102.0,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,120000.0,26,-1,2,0,0,0,2,2682.0,1725.0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,90000.0,34,0,0,0,0,0,0,29239.0,14027.0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,50000.0,37,0,0,0,0,0,0,46990.0,48233.0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,50000.0,57,-1,0,-1,0,0,0,8617.0,5670.0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000.0,39,0,0,0,0,0,0,188948.0,192815.0,...,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,150000.0,43,-1,-1,-1,-1,0,0,1683.0,1828.0,...,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,30000.0,37,4,3,2,-1,0,0,3565.0,3356.0,...,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,80000.0,41,1,-1,0,0,0,-1,-1645.0,78379.0,...,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


In [10]:
df['default.payment.next.month'].value_counts()

default.payment.next.month
0    23364
1     6636
Name: count, dtype: int64

In [11]:
df1 = df[df['default.payment.next.month']==1]
df2 = df[df['default.payment.next.month']==0].sample(6636)

final_df = pd.concat([df1, df2])

In [12]:
final_df['default.payment.next.month'].value_counts()

default.payment.next.month
1    6636
0    6636
Name: count, dtype: int64

In [16]:
final_df_shfl = final_df.sample(final_df.shape[0])

In [17]:
final_df_shfl

,LIMIT_BAL,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
13071,30000.0,25,0,0,2,0,0,2,7527.0,9705.0,...,9333.0,9505.0,8699.0,2700.0,0.0,1000.0,700.0,0.0,784.0,1
24263,400000.0,29,0,0,0,0,0,0,237425.0,242225.0,...,252429.0,257510.0,262931.0,8600.0,8000.0,9000.0,8200.0,8500.0,8500.0,1
13398,100000.0,50,0,0,0,0,0,0,95889.0,96791.0,...,69166.0,53270.0,-1465.0,4200.0,2785.0,2291.0,1965.0,1465.0,62121.0,0
9509,70000.0,31,0,0,0,0,0,0,63441.0,29091.0,...,30395.0,31034.0,31668.0,1789.0,1500.0,1089.0,1130.0,1150.0,1200.0,0
10853,120000.0,28,0,0,0,2,0,0,75982.0,78213.0,...,78273.0,79914.0,81514.0,3451.0,6275.0,0.0,2898.0,2926.0,6210.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7922,50000.0,30,0,0,0,2,2,2,60762.0,58791.0,...,55405.0,55904.0,52160.0,2300.0,4400.0,0.0,4000.0,0.0,4000.0,1
5501,290000.0,55,1,-2,-1,-1,-1,2,0.0,166.0,...,-150.0,1134.0,777.0,166.0,2462.0,150.0,1833.0,0.0,500.0,1
1614,360000.0,44,0,0,0,0,2,0,270248.0,138566.0,...,149758.0,147152.0,149460.0,5000.0,5000.0,8600.0,0.0,5000.0,5000.0,0
3481,40000.0,23,2,2,2,0,0,0,19827.0,20709.0,...,21420.0,23066.0,23683.0,1500.0,1000.0,1000.0,2000.0,1000.0,894.0,1


In [18]:
final_df_shfl.to_csv('/Users/dalibor/VsCodeProjects/imggi-course/notebooks/ml/data/UCI_Credit_Card_12k_samples.csv', index=False)